In [1]:
from pathlib import Path

import numpy as np
from astropy.table import Table, vstack, unique
from tqdm.auto import tqdm


def merge_ecsv_files(
    input_dir,
    output_file,
    duplicate_column="source_id"
):


    input_dir = Path(input_dir)
    output_file = Path(output_file)

    files = sorted(
        input_dir.rglob("*.ecsv")
    )

    print(f"Archivos encontrados: {len(files)}")

    if len(files) == 0:
        raise ValueError(
            f"No se han encontrado archivos ECSV en {input_dir}"
        )

    # Evitar volver a leer el archivo final si está dentro
    # de la misma carpeta
    files = [
        file
        for file in files
        if file.resolve() != output_file.resolve()
    ]

    tables = []

    for file in tqdm(
        files,
        desc="Leyendo ECSV"
    ):

        try:

            table = Table.read(
                file,
                format="ascii.ecsv"
            )

            if len(table) > 0:
                tables.append(table)

        except Exception as exc:

            print(
                f"\nError leyendo {file}: {exc}"
            )

    if len(tables) == 0:
        raise ValueError(
            "No se ha podido cargar ninguna tabla."
        )

    print("\nUniendo tablas...")

    combined = vstack(
        tables,
        metadata_conflicts="silent"
    )

    print(
        f"Filas antes de eliminar duplicados: "
        f"{len(combined)}"
    )

    if duplicate_column not in combined.colnames:

        raise ValueError(
            f"No existe la columna '{duplicate_column}'.\n"
            f"Columnas disponibles: {combined.colnames}"
        )

    combined = unique(
        combined,
        keys=duplicate_column,
        keep="first"
    )

    print(
        f"Filas después de eliminar duplicados: "
        f"{len(combined)}"
    )

  
    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    combined.write(
        output_file,
        format="ascii.ecsv",
        overwrite=True
    )

    print(
        f"\nArchivo guardado en:\n{output_file}"
    )

    return combined

In [2]:
input_dir = (
    "/Users/carlasequero/Desktop/trabajo-fin-master/solution/data/HR_data"
)

In [3]:
output_file = (
    "/Users/carlasequero/Desktop/trabajo-fin-master/solution/data/"
    "gaia_data.ecsv"
)

In [ ]:
gaia_hr_table = merge_ecsv_files(
    input_dir=input_dir,
    output_file=output_file,
    duplicate_column="source_id"
)

Archivos encontrados: 18899


Leyendo ECSV:   0%|          | 0/18899 [00:00<?, ?it/s]